# Using the OpenAlex API

##  🎓 About OpenAlex

[OpenAlex](https://openalex.org/) is a great resource for searching and accessing **scholarly publication data** comprehensively. Furthermore, OpenAlex holds data about **authors, institutions and sources** and more [entities](https://docs.openalex.org/api-entities/entities-overview) relating to those publications. For this, it brings together data from a series well-known [data sources](https://docs.openalex.org/additional-help/faq#where-does-your-data-come-from).

And, what's maybe the best about it: OpenAlex data is published under the Creative Commons Public Domain Dedication ([CC0](https://creativecommons.org/publicdomain/zero/1.0/)) - waiving copyright at all!

**💻 Snapshots and API**

OpenAlex offers data [snapshots](https://docs.openalex.org/download-snapshot) on a regularly basis, but can accessed far more nuanced via [**REST API**](https://docs.openalex.org/how-to-use-the-api/api-overview). This notebook will give you a small overview about its basic functionality and the data which can be accessed there.

There is one thing which is helpful to understand and use OpenAlex' API: It consists of different endpoints representing the different entity types, OpenAlex provides data on:
* `/works` - scholarly publications like journal articles, book chapters etc.
* `/authors` - the contributors of these publications
* `/topics` - the subjects, which are treated by works
* `/institutions` - the institutions, which host contributors of publications like universities
* `/sources` - the sources which hold the single works, like journals, series, conferences, and repositories
* `/publishers`- companies and organizations that distribute works
* `/funders`- organizations that fund research


This notebook will show these endpoints, but not all possible actions. Despite this, OpenAlex API is constantly growing: To get it all, consult the [documentation](https://developers.openalex.org/).

**☝️ Caveat**

Gathering and integrating data of scholarly publications is a tough thing to do. Data sources are manifold and even original data is often incomplete. So, also data of aggregators like OpenAlex will not be 100% complete, free from errors or duplicates. **Handle all derivative numbers with care** and draw conclusions with respect to this imperfect data base. If possible, use other data sources to crosscheck your findings.

**🔑 Get a (free) API key**

Since **13 February 2026** the OpenAlex API requires an **API key** for all requests. The former "polite pool" based on the `mailto` parameter has been retired - the parameter is simply ignored now. The key is free: create an account and get your key at [openalex.org/settings/api](https://openalex.org/settings/api). It comes with a free daily usage budget (resetting at midnight UTC). See [Authentication](https://help.openalex.org/api/authentication/) and [Pricing](https://help.openalex.org/access/pricing/) for details.

**🗂️ Data version "Walden"**

Since November 2025 the API serves the rebuilt data version "Walden". Coverage and data quality improved, some fields were removed or deprecated (e.g. *Concepts* in favour of *Topics*, `last_known_institution` in favour of `last_known_institutions`), and new work types like `preprint` or `review` were introduced. Additional "expansion" records (mostly datasets and repository records) are excluded from API results by default. Numbers shown in this notebook may therefore differ from earlier versions. Check the [deprecations](https://developers.openalex.org/guides/deprecations) page for details.

In [1]:
# Check Python version
!python --version

Python 3.13.17


In [2]:
# Install necessary libraries if not already available
#!pip install requests pandas

In [3]:
# Import the necessary libraries
import json, os, requests
import pandas as pd

In [4]:
# IMPORTANT: Provide your OpenAlex API key (https://openalex.org/settings/api)
# Preferably set it as environment variable OPENALEX_API_KEY instead of writing it into the notebook
api_key = os.environ.get('OPENALEX_API_KEY', '...')

## 📚 Works endpoint

The core of OpenAlex data are **scholarly publications**. These are are represented as ***Work* entities**, which can be retrieved via the `/works` endpoint. Single *Work* entities can be directly accessed by their OpenAlex identifier (starting with a "W" like `W3093430496`) or by their Digital Object Identifier (DOI) respectively. You can access a random publication with `https://api.openalex.org/works/random`.

To learn about the attributes a *Work* entity can have, take a look into the [Work](https://docs.openalex.org/about-the-data/work) object documentation.

We'll start by accessing a single *Work* entity aka a publication record. The code cell below queries the OpenAlex API (via it's `/works` endpoint). The base URL is completed with an identifier and enriched with your API key.

There are several examples you might try by commenting out the identifiers and adapt the request accordingly.

In [5]:
base_url = 'https://api.openalex.org/works/'
params = {'api_key': api_key}

# Using different work identifiers
openalex_id = 'W3093430496'                               # doctoral thesis
#openalex_id = 'W2489129678'                              # monograph
#openalex_id = 'W4200635144'                             # conference publication
#doi = 'doi:10.1007/978-3-319-20319-5_4'                  # book part
#doi = 'doi:10.1007/10.22617/tim210529'                   # report

doi =  'doi:https://doi.org/10.1007/s10479-016-2314-1'    # journal article (formerly green OA, now closed)
#doi = 'doi:https://doi.org/10.1371/journal.pone.0121874' # gold OA article
#doi = 'doi:https://doi.org/10.1007/s10489-020-02029-z'   # bronze OA article
#doi = 'https://doi.org/10.1134/s2079086415010053'         # closed article

    
# Adapt identifier if necessary: doi, openalex_id
r = requests.get(base_url + doi, params=params)
data = r.json()
type(data)

dict

The data answer of the OpenAlex API is [JSON](https://en.wikipedia.org/wiki/JSON) formatted. Therefore, it is decoded with the `json()` command and written into an Python object named `data`. Resembling the JSON structure, this is a Python [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries).

This request and decoding method will be used throughout the whole notebook. If a request fails, check `r.status_code` and `r.text` first - e.g. a missing or invalid API key or an exhausted daily budget.

The `list()` function gives you an overview of the available data fields (which are the *Work*'s attributes) in the dictionary by listing all its *keys*. Calling a certain attribute *value* can easily done with its key, for instance `data['id']`.

In [6]:
# Overview of available data fields
list(data)

['id',
 'doi',
 'title',
 'display_name',
 'publication_year',
 'publication_date',
 'ids',
 'language',
 'primary_location',
 'volume',
 'type',
 'indexed_in',
 'open_access',
 'authorships',
 'institutions',
 'countries_distinct_count',
 'institutions_distinct_count',
 'corresponding_author_ids',
 'corresponding_institution_ids',
 'apc_list',
 'apc_paid',
 'fwci',
 'has_fulltext',
 'cited_by_count',
 'citation_normalized_percentile',
 'cited_by_percentile_year',
 'biblio',
 'is_retracted',
 'is_paratext',
 'is_xpac',
 'primary_topic',
 'topics',
 'keywords',
 'concepts',
 'mesh',
 'locations_count',
 'locations',
 'best_oa_location',
 'sustainable_development_goals',
 'sustainable_development_goals_aurora',
 'study_designs',
 'awards',
 'funders',
 'has_content',
 'content_urls',
 'referenced_works_count',
 'referenced_works',
 'related_works',
 'abstract_inverted_index',
 'counts_by_year',
 'updated_date',
 'created_date']

In [7]:
# Calling the OpenAlex work ID field by its key ("id")
data['id']

'https://openalex.org/W2518998880'

In [8]:
# Publication title
data['title']

'A framework for investigating optimization of service parts performance with big data'

In [9]:
# All available identifiers, given as an own Python dictionary
data['ids']

{'openalex': 'https://openalex.org/W2518998880',
 'doi': 'https://doi.org/10.1007/s10479-016-2314-1',
 'mag': '2518998880'}

In [10]:
# Publication type
data['type']

'article'

In [11]:
# Bibliographic information
data['biblio']

{'volume': '270', 'issue': '1-2', 'first_page': '65', 'last_page': '74'}

In [12]:
# Full text availability: these fields changed with the "Walden" data version,
# so we read them with .get() to avoid errors if a field is not delivered anymore
{key: data.get(key) for key in ['has_fulltext', 'has_content', 'fulltext_origin']}

{'has_fulltext': False,
 'has_content': {'grobid_xml': False, 'pdf': False},
 'fulltext_origin': None}

In [13]:
# Bibliographic information of primary location (= publication medium/infrastructure)
# "issn_l" should be used preferably of all ISSNs while using OpenAlex API with ISSN identifier
data['primary_location']

{'id': 'doi:10.1007/s10479-016-2314-1',
 'doi': 'https://doi.org/10.1007/s10479-016-2314-1',
 'is_oa': False,
 'landing_page_url': 'https://doi.org/10.1007/s10479-016-2314-1',
 'pdf_url': None,
 'source': {'id': 'https://openalex.org/S57667410',
  'display_name': 'Annals of Operations Research',
  'issn_l': '0254-5330',
  'issn': ['0254-5330', '1572-9338'],
  'is_oa': False,
  'is_in_doaj': False,
  'is_core': True,
  'listed_in': ['abdc-a',
   'anvur-class-a',
   'anvur-scientific',
   'cwts-core',
   'fnege-2',
   'jufo-1',
   'ki-jl-1',
   'norway-1',
   'poland-70',
   'russia-white-list-1',
   'vabb-shw'],
  'host_organization': 'https://openalex.org/P4310319900',
  'host_organization_name': 'Springer Science+Business Media',
  'host_organization_lineage': ['https://openalex.org/P4310319900',
   'https://openalex.org/P4310319965'],
  'host_organization_lineage_names': ['Springer Science+Business Media',
   'Springer Nature'],
  'type': 'journal'},
 'license': None,
 'license_id': 

In [14]:
# Alternative locations, especially to provide Open Access versions of closed publications
data['locations']

[{'id': 'doi:10.1007/s10479-016-2314-1',
  'doi': 'https://doi.org/10.1007/s10479-016-2314-1',
  'is_oa': False,
  'landing_page_url': 'https://doi.org/10.1007/s10479-016-2314-1',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S57667410',
   'display_name': 'Annals of Operations Research',
   'issn_l': '0254-5330',
   'issn': ['0254-5330', '1572-9338'],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': True,
   'listed_in': ['abdc-a',
    'anvur-class-a',
    'anvur-scientific',
    'cwts-core',
    'fnege-2',
    'jufo-1',
    'ki-jl-1',
    'norway-1',
    'poland-70',
    'russia-white-list-1',
    'vabb-shw'],
   'host_organization': 'https://openalex.org/P4310319900',
   'host_organization_name': 'Springer Science+Business Media',
   'host_organization_lineage': ['https://openalex.org/P4310319900',
    'https://openalex.org/P4310319965'],
   'host_organization_lineage_names': ['Springer Science+Business Media',
    'Springer Nature'],
   'type': 'journal'},
  '

In [15]:
# Open Access status
data['open_access']

{'is_oa': False,
 'oa_status': 'closed',
 'oa_url': None,
 'any_repository_has_fulltext': False}

In [16]:
# The first 5 cited works by their OpenAlex links
data['referenced_works'][:5]

['https://openalex.org/W581472570',
 'https://openalex.org/W1215384480',
 'https://openalex.org/W1486780809',
 'https://openalex.org/W1558765224',
 'https://openalex.org/W1570319579']

In [17]:
# Times cited
data['cited_by_count']

24

In [18]:
# Citations by year (since 2012)
data['counts_by_year']

[{'year': 2025, 'cited_by_count': 1},
 {'year': 2024, 'cited_by_count': 1},
 {'year': 2023, 'cited_by_count': 2},
 {'year': 2022, 'cited_by_count': 5},
 {'year': 2021, 'cited_by_count': 2},
 {'year': 2020, 'cited_by_count': 3},
 {'year': 2019, 'cited_by_count': 4},
 {'year': 2017, 'cited_by_count': 6}]

In [19]:
# API link to citing publications (built with the "cites" filter of the works endpoint)
'https://api.openalex.org/works?filter=cites:' + data['id'].split('/')[-1]

'https://api.openalex.org/works?filter=cites:W2518998880'

In [20]:
# Status information: paratext, retraction
print(data['is_paratext'])
data['is_retracted']

False


False

A special feature of OpenAlex *Work* data is the attribute `abstract_inverted_index`. Instead of delivering the plain text **abstract of a publication**, its vocabulary or tokenized version is given, listing all words/tokens with their position in the abstract. This is done for copyright reasons. However, this information can be used to search *Works* (see 2.2) and build other applications upon.
Note that punctuation is treated as part of the tokens.

In [21]:
# Inverted index with positions of words
data['abstract_inverted_index']

In [22]:
# First 5 related works by their OpenAlex links
data['related_works'][:5]

['https://openalex.org/W2016028516',
 'https://openalex.org/W2133079869',
 'https://openalex.org/W2530755953',
 'https://openalex.org/W2549510740',
 'https://openalex.org/W2808736867']

In [23]:
# How up to date is this information?
data['updated_date']

'2026-10-16T09:59:22.235173'

In [24]:
# Primary topic (subject) with its hierarchy of subfield, field and domain
data['primary_topic']

{'id': 'https://openalex.org/T10328',
 'display_name': 'Supply Chain and Inventory Management',
 'score': 0.8544999957084656,
 'subfield': {'id': 'https://openalex.org/subfields/1404',
  'display_name': 'Management Information Systems'},
 'field': {'id': 'https://openalex.org/fields/14',
  'display_name': 'Business, Management and Accounting'},
 'domain': {'id': 'https://openalex.org/domains/2',
  'display_name': 'Social Sciences'}}

In [25]:
# Use Python list comprehension to access the display names and scores of all attached topics
[[i['display_name'], i['score']] for i in data['topics']]

[['Supply Chain and Inventory Management', 0.8544999957084656],
 ['Big Data and Business Intelligence', 0.11089999973773956],
 ['Sustainable Supply Chain Management', 0.010499999858438969]]

In [26]:
# How many MESH descriptors are there?
len(data['mesh'])

0

In [27]:
# Show the first 5 MESH descriptors with qualifier ("None" = no qualifier)
[[i['descriptor_name'], i['qualifier_name']] for i in data['mesh'][:5]]

[]

In [28]:
# Showing the APC of the source journal
data['apc_list']

{'value': 3090, 'currency': 'USD', 'value_usd': 3090}

### Author and affiliation data

The *Works* object can hold attributes which provide more detailed information, or in a certain way, sub-attributes, like the attribute `authorships`.
Let's take a look at some `authorships` data: It holds all authors of the publication, each introduced by the `author_position` statement. Thereafter follows the individual author's data, resembling the *Author* entity will get to now later with its own endpoint.

Note that authors can have multiple `institutions` (which are their affiliations), each in turn described in detail, too.

In [29]:
# Showing the first 2 authors, assembled in "authorships" attribute
data['authorships'][:2]

[{'author_position': 'first',
  'author': {'id': 'https://openalex.org/A5080205434',
   'display_name': 'Christopher A. Boone',
   'orcid': 'https://orcid.org/0000-0001-9654-9062',
   'observed_orcids': ['https://orcid.org/0000-0001-9654-9062']},
  'institutions': [{'id': 'https://openalex.org/I128956969',
    'display_name': 'Texas Christian University',
    'ror': 'https://ror.org/054b0b564',
    'country_code': 'US',
    'type': 'education',
    'lineage': ['https://openalex.org/I128956969']}],
  'countries': ['US'],
  'is_corresponding': False,
  'raw_author_name': 'Christopher A. Boone',
  'raw_affiliation_strings': ['Neeley School of Business, Texas Christian University, Fort Worth, TX, USA'],
  'raw_orcid': None,
  'affiliations': [{'raw_affiliation_string': 'Neeley School of Business, Texas Christian University, Fort Worth, TX, USA',
    'institution_ids': ['https://openalex.org/I128956969']}]},
 {'author_position': 'middle',
  'author': {'id': 'https://openalex.org/A5020324702

In [30]:
# Print out the names, ORCIDs, raw affiliation strings (a list) and positions of all authors with a loop
for i in data['authorships']:
    print(i['author']['display_name'])
    print(i['author']['orcid'])
    print(i['raw_affiliation_strings'])
    print(i['author_position'])
    print('---')

Christopher A. Boone
https://orcid.org/0000-0001-9654-9062
['Neeley School of Business, Texas Christian University, Fort Worth, TX, USA']
first
---
Benjamin Thomas Hazen
https://orcid.org/0000-0003-4891-3071
['Department of Operational Sciences, Air Force Institute of Technology, Wright Patterson AFB, OH, USA']
middle
---
Joseph B. Skipper
None
['Stafford School of Business, Abraham Baldwin Agricultural College, Tifton, GA, USA']
middle
---
Robert E. Overstreet
https://orcid.org/0000-0002-5047-2415
['Department of Supply Chain and Information Systems, Iowa State University, Ames, IA, USA']
last
---


In [31]:
# Print out all affiliations (short, normalized form) of the authors with a nested loop
for i in data['authorships']:
    for j in i['institutions']:
        print(j['display_name'])

Texas Christian University
U.S. Air Force Institute of Technology
Abraham Baldwin Agricultural College
Iowa State University


### Use the search parameter

With a slight adjustment you can use the *Works* endpoint for a simple **search in the title, abstract fields** and **fulltext**, if available for OpenALex. You can search multiple terms just by separating them with a blank. You can also search for **exact phrases** using quotation marks.

The ranking (`relevance_score`) depends on similarity between request and field data, proximity of the single terms and citation counts. See [here](https://docs.openalex.org/api-entities/works/search-works) for a more detailed description of the search functionality.

In [32]:
# Search 2 terms in title, abstract and fulltext, where available
url = 'https://api.openalex.org/works?search=einstein philosoph'

# Compare
#url = 'https://api.openalex.org/works?search=einsteins philosophie'

# Search an exact phrase using quotes ""
#url = 'https://api.openalex.org/works?search="einstein als philosoph"'

params = {'api_key': api_key}

r = requests.get(url, params=params)
data = r.json()

In [33]:
data['meta']

{'count': 1627,
 'db_response_time_ms': 33,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get works with (einstein philosoph) in the full text',
  'x_new_oql': 'get works with (einstein philosoph) in the full text',
  'oqo': {'get_rows': 'works',
   'filter_rows': [{'column_id': 'fulltext.search',
     'value': 'einstein philosoph',
     'operator': 'has'}]},
  'url': '/works?filter=fulltext.search:einstein philosoph',
  'order': 'best matches first',
  'order_default': True},
 'cost_usd': 0.001}

In [34]:
for i in data['results'][:10]:
    print(i['title'])
    print(i['relevance_score'])
    print(i['type'])
    print('---')

Einsteins Philosoph: Moritz Schlick und die Relativitätstheorie
144.35924
other
---
Albert Einstein als Philosoph und Naturforscher
93.725914
book
---
Albert Einstein als Philosoph und Naturforscher
83.28591
book
---
Teil V: Schlussbemerkungen: Einstein – Wissenschaftler und Philosoph
76.955505
book-chapter
---
Albert Einstein als Philosoph und Naturforscher : eine Auswahl
73.94067
article
---
Schill, P. A. : Albert Einstein als Philosoph und Naturforscher
73.2908
article
---
Der Philosoph als Schatzgraber. Moritz Schlick und das „Weltbild“ der Relativitatstheorie Einsteins.
61.80072
conference-paper
---
P. A. Schilpp (Ed.): Albert Einstein als Philosoph und Naturforscher. Eine Auswahl. Friedr. Vieweg & Sohn, Braunschweig/Wiesbaden 1983. 250 Seiten, Preis: DM 34,—
43.813385
article
---
Landolt‐Börnstein: Molekelkonstanten, Elektronenresonanz‐Spektroskopie, Ergänzung der Bände II/4 und II/6/Pais: „Subtle is the Lord …” The Science and Life of Albert Einstein/Schilpp: Albert Einstein als

### Access multiple Work entities

After retrieving one specific publication record successfully, it's time to look for a bunch of publications. First of all, it is easy to **access all available works** with the OpenAlex API - just use the *Works* endpoint without any further parameters. But presumably you will need this amount of data only in exceptional cases.

But you can easily **add** to the *Work* endpoint **several filters** to narrow down your search. You find the **available filters** for *Work* entities [here](https://docs.openalex.org/api-entities/works/filter-works). As you may notice these are mainly the attributes and sub-attributes of the *Work* entities seen above.

#### Filter works by publication type

Let's start with filtering all available publications due to a **certain publication type**, like for instance book parts or reports. OpenAlex uses its own [list of work types](https://api.openalex.org/work-types) (e.g. `article`, `book-chapter`, `dissertation`, `preprint`, `review`); the original Crossref type is still available in the attribute `type_crossref`. Feel free to try them out in the  code cell below.

In [35]:
# Accessing all publication of a certain publication type
#url = 'https://api.openalex.org/works'   # all available publications

url = 'https://api.openalex.org/works?filter=type:dissertation'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()

In [36]:
print(list(data))
data['meta']

['meta', 'results', 'group_by']


{'count': 11415055,
 'db_response_time_ms': 31,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get dissertations',
  'x_new_oql': 'get dissertations',
  'oqo': {'get_rows': 'works',
   'filter_rows': [{'column_id': 'type', 'value': 'dissertation'}]},
  'url': '/works?filter=type:dissertation',
  'order': 'most cited (age-adjusted) first',
  'order_default': True},
 'cost_usd': 0.0001}

In the `results` section you will find the matching records.

The `count` attribute in the `meta`section says how many results are there at all. The `per_page` attribute tells you how many records are delivered at once as default.

In [37]:
# Pick one example from the results list
data['results'][11]

{'id': 'https://openalex.org/W3121689684',
 'doi': 'https://doi.org/10.1016/j.jpubeco.2010.11.006',
 'title': 'Counting and Multidimensional Poverty Measurement',
 'display_name': 'Counting and Multidimensional Poverty Measurement',
 'publication_year': 2017,
 'publication_date': '2017-01-01',
 'ids': {'openalex': 'https://openalex.org/W3121689684',
  'doi': 'https://doi.org/10.1016/j.jpubeco.2010.11.006',
  'mag': '3121689684'},
 'language': 'en',
 'primary_location': {'id': 'pmh:oai:intellectum.unisabana.edu.co:10818/31136',
  'doi': 'https://doi.org/10.1016/j.jpubeco.2010.11.006',
  'is_oa': True,
  'landing_page_url': 'http://doi.org/10.1016/j.jpubeco.2010.11.006',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S4306401748',
   'display_name': 'Intellectum (Universidad de La Sabana)',
   'issn_l': None,
   'issn': None,
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'listed_in': [],
   'host_organization': 'https://openalex.org/I157650460',
   'host

#### Filter works by "source":  journal, repository etc

Next we try to retrieve all publications of a certain "source". ***Source* entities** in OpenAlex are the venues, like **journals, conferences, preprint repositories,** and **institutional repositories**, that host publications. For journals articles this will be their journals, for book parts their volume. Sources can also be monograph series like [S4210167446](https://api.openalex.org/sources/S4210167446), repositories like [S4306402567](https://api.openalex.org/sources/S4306402567).

Source information is given in the *Works* attribute `primary_location` and `locations`. To get an impression of sources you may use the random generation `https://api.openalex.org/sources/random`.

To identify the desired source you can use its OpenAlex ID (starting with a "S"), for instance `S4210209919`. This identifier is also a sub-attribute of `locations.source`, and can be addressed with `locations.source.id` in the *Works* endpoint filter. You may also use the popular [ISSN](https://en.wikipedia.org/wiki/International_Standard_Serial_Number) identifier in the same way.

In [38]:
# Accessing works of a certain host venue, using OpenAlex ID
url = 'https://api.openalex.org/works?filter=locations.source.id:S4210209919'

# Using ISSN identifier
#url = 'https://api.openalex.org/works?filter=locations.source.issn:2391-7652'

params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 212,
 'db_response_time_ms': 33,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get works where any location source is (S4210209919)',
  'x_new_oql': 'get works where any location source is (S4210209919)',
  'oqo': {'get_rows': 'works',
   'filter_rows': [{'column_id': 'locations.source.id',
     'value': 'S4210209919'}]},
  'url': '/works?filter=locations.source.id:S4210209919',
  'order': 'most cited (age-adjusted) first',
  'order_default': True},
 'cost_usd': 0.0001}

In [39]:
# Article titles of the result set
[i['title'] for i in data['results']]

['Caste: The Origins of Our Discontents',
 "'Dalit Feminist Theory: A Reader'",
 'Dalit or Brahmanical Patriarchy? Rethinking Indian Feminism',
 'Dalits and the Making of Modern India',
 'Caste Identities and Structures of Threats: Stigma, Prejudice and Social Representation in Indian universities',
 'Caste-ing Space: Mapping the Dynamics of Untouchability in Rural Bihar, India',
 'Identity of Language and the Language of Erasure',
 'Caste and Socioeconomic Inequality in Child Health and Nutrition in India: Evidences from National Family Health Survey',
 'Manual Scavenging in India: The Banality of An Everyday Crime',
 'Caste and Counselling Psychology in India: Dalit Perspectives in Theory and Practice',
 'Indian Media and Caste: Of Politics, Portrayals and Beyond',
 'Witch Hunting: A Form of violence against Dalit Women in India',
 'Health Disparity and Health Equity in India: Understanding the Difference and the Pathways Towards Policy',
 'Caste and Religion Matters in Access to Hou

We might adapt now the request by defining a **bigger default result set** to get all hits at once.

Instead to add the parameter to the URL directly, you may also use the `params` dictionary for more clarity.

In [40]:
url = 'https://api.openalex.org/works?'

params = {'api_key': api_key,
          'filter': 'locations.source.id:S4210209919',
          'per_page': 100}

r = requests.get(url, params=params) 
data = r.json()

In [41]:
data['meta']

{'count': 212,
 'db_response_time_ms': 43,
 'page': 1,
 'per_page': 100,
 'groups_count': None,
 'x_query': {'oql': 'get works where any location source is (S4210209919)',
  'x_new_oql': 'get works where any location source is (S4210209919)',
  'oqo': {'get_rows': 'works',
   'filter_rows': [{'column_id': 'locations.source.id',
     'value': 'S4210209919'}]},
  'url': '/works?filter=locations.source.id:S4210209919&per_page=100',
  'order': 'most cited (age-adjusted) first',
  'order_default': True},
 'cost_usd': 0.0001}

In [42]:
# Pick one example of the result set
data['results'][67]

{'id': 'https://openalex.org/W4206408406',
 'doi': 'https://doi.org/10.26812/caste.v2i2.218',
 'title': 'On the margins of Healthcare: Role of Social Capital in Health of Migrants in India',
 'display_name': 'On the margins of Healthcare: Role of Social Capital in Health of Migrants in India',
 'publication_year': 2021,
 'publication_date': '2021-12-18',
 'ids': {'openalex': 'https://openalex.org/W4206408406',
  'doi': 'https://doi.org/10.26812/caste.v2i2.218'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.26812/caste.v2i2.218',
  'doi': 'https://doi.org/10.26812/caste.v2i2.218',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.26812/caste.v2i2.218',
  'pdf_url': 'https://journals.library.brandeis.edu/index.php/caste/article/download/218/79',
  'source': {'id': 'https://openalex.org/S4210209919',
   'display_name': 'CASTE / A Global Journal on Social Exclusion',
   'issn_l': '2639-4928',
   'issn': ['2639-4928'],
   'is_oa': True,
   'is_in_doaj': True,
   'is_core'

In [43]:
# Access 5 publications by title
[i['title'] for i in data['results'][35:40]]

['Intersecting Dalit and Cultural Studies: De-brahmanising the Disciplinary Space',
 'Revolt of the Upper Castes',
 'Persisting Prejudice: Measuring Attitudes and Outcomes by Caste and Gender',
 "'Ambedkar's Constitution': A Radical Phenomenon in Anti-Caste Discourse?",
 'Caste and Consequences']

In [44]:
# Access the topics of a sample from the middle of the publication set
[[j['display_name'] for j in i['topics']] for i in data['results'][35:37]]

[['South Asian Studies and Diaspora',
  'South Asian Cinema and Culture',
  'Indian and Buddhist Studies'],
 ['South Asian Studies and Diaspora',
  'Indian Economic and Social Development',
  'South Asian Studies and Conflicts']]

#### Filter works by author & handle big result sets

Similarly to the venue filtering, you may show all publication by a certain author, using its OpenAlex ID (starting with a "A"), for instance `A2420227930`. This identifier can be addressed with `authorships.author.id` or shorter, `author.id` in the *Works* endpoint filter.

Furthermore, you may access a certain sub result set ("result page") via the `page` parameter, like in the example below.

In [45]:
url = 'https://api.openalex.org/works?filter=author.id:A5067955208&page=1&per-page=50'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 370,
 'db_response_time_ms': 18,
 'page': 1,
 'per_page': 50,
 'groups_count': None,
 'x_query': {'oql': "get (A5067955208)'s works",
  'x_new_oql': "get (A5067955208)'s works",
  'oqo': {'get_rows': 'works',
   'filter_rows': [{'column_id': 'authorships.author.id',
     'value': 'A5067955208'}]},
  'url': '/works?filter=authorships.author.id:A5067955208&per_page=50&page=1',
  'order': 'most cited (age-adjusted) first',
  'order_default': True},
 'cost_usd': 0.0001}

Since the `per-page` parameter only can be set up to 200, several requests are necessary to access the whole result set.

This can be easily done by writing a small loop.

In [46]:
results = []
for page in range(1,5):
    url = 'https://api.openalex.org/works?'
    params = {'api_key': api_key,
              'filter': 'author.id:A5067955208',
              'per-page': 100,
              'page': page}
    r = requests.get(url, params=params)
    data = r.json()
    for record in data['results']:
        results.append(record)
len(results)

370

In [47]:
[i['title'] for i in results[320:330]]

['P4.13 VITAMIN D SUPPLEMENTATION IMPROVES ENDOTHELIAL FUNCTION IN TYPE 2 DIABETES – A RANDOMIZED CONTROLLED TRIAL',
 'Impaired bone material properties increase the risk of all fractures and severity of vertebral fractures in osteoporosis: an impact microindentation study',
 '[Osteonecrosis of the jaw].',
 'Impact of 3-year vitamin D and calcium supplementation on mineral and organic matrix formation of trabecular bone in postmenopausal osteoporosis',
 '[D-vitamin: old paradoxes and new perspectives].',
 'Re: Femurfraktur og kjeveleddsdestruksjon etter bruk av bisfosfonater',
 'Remodellierung und Knochenstruktur bei Osteoporose',
 'ZOLEDRONIC ACID IMPROVES HEALTH-RELATED QUALITY OF LIFE IN PATIENTS WITH HIP FRACTURE: RESULTS OF HORIZON-RFT',
 'Clinical updates in hypoparathyroisism',
 'Çene kemiğinin osteonekrozu ve osteoporoz için bifosfonat tedavisi']

#### Group works due to a certain feature

Grouping is another way to adjust your data search. Here, you **split the result set** into groups **according to a certain data attribute**. Note that doing so, you are not targeting the records themselves, but deriving statements from the data. Because of this you will not use `filter` as parameter but `group_by` in your request. Accordingly, you will find the `results` section empty, but your demanded data in the `group_by`section.

For example you might look for the hosting sources of all publications, grouping them around this feature, and get the size of these groups. Note that the maximum of groups is 200 - if there are more variants you will get the 200 biggest of them.

In [48]:
url = 'https://api.openalex.org/works'
params = {'api_key': api_key,
         'group_by': 'locations.source.id'}

r = requests.get(url, params=params) 
data = r.json()

In [49]:
list(data)

['meta', 'group_by']

In [50]:
data['meta']

{'count': 328028539,
 'db_response_time_ms': 1246,
 'page': 1,
 'per_page': 200,
 'groups_count': 200,
 'x_query': {'oql': 'get works; then, group those works by any location source',
  'x_new_oql': 'get works; then, group those works by any location source',
  'oqo': {'get_rows': 'works',
   'group_by': [{'column_id': 'locations.source.id'}]},
  'url': '/works?group_by=locations.source.id',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [51]:
data['group_by'][:5]

[{'key': 'https://openalex.org/S4306525036',
  'key_display_name': 'PubMed',
  'count': 40998069},
 {'key': 'https://openalex.org/S7407052672',
  'key_display_name': 'NIFS',
  'count': 14985209},
 {'key': 'https://openalex.org/S4306401280',
  'key_display_name': 'DOAJ (DOAJ: Directory of Open Access Journals)',
  'count': 14015290},
 {'key': 'https://openalex.org/S2764455111',
  'key_display_name': 'PubMed Central',
  'count': 11853512},
 {'key': 'https://openalex.org/S7407053449',
  'key_display_name': 'DiSSCo',
  'count': 6829110}]

In [52]:
# Accessing the biggest 20 sources, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:20]]

[['PubMed', 40998069],
 ['NIFS', 14985209],
 ['DOAJ (DOAJ: Directory of Open Access Journals)', 14015290],
 ['PubMed Central', 11853512],
 ['DiSSCo', 6829110],
 ['HAL', 4116465],
 ['RePEc: Research Papers in Economics', 3906752],
 ['Institutional Repositories DataBase (IRDB)', 3730426],
 ['Zenodo (CERN European Organization for Nuclear Research)', 3682931],
 ['Europe PMC (PubMed Central)', 3638975],
 ['Springer eBooks', 3444613],
 ['arXiv (Cornell University)', 3386489],
 ['LA Referencia (Red Federada de Repositorios Institucionales de Publicaciones Científicas)',
  3156087],
 ['Dialnet (Universidad de la Rioja)', 2297737],
 ['Global Biodiversity Information Facility', 1977857],
 ['Routledge eBooks', 1904156],
 ['SSRN Electronic Journal', 1832687],
 ['DOE Pacific Northwest National Laboratory (PNNL) Repository', 1832602],
 ['Figshare', 1815262],
 ['OSTI OAI (U.S. Department of Energy Office of Scientific and Technical Information)',
  1787812]]

#### Combine filters and grouping

You can easily combine filters and grouping, for instance, filter according to a specific institution, like *University of Bern* `I118564535`, and group its publications by their main subject `primary_topic.id`. Then, of course, the resulting groups are based on the filtered subset.

In [53]:
# Use the "params" dictionary for using filter and group_by at the same time
url = 'https://api.openalex.org/works'
params = {'api_key': api_key,
          'filter': 'institutions.id:I118564535',
          'group_by': 'primary_topic.id'}

r = requests.get(url, params=params)
data = r.json()

In [54]:
# Accessing the 20 most frequent topics, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:20]]

[['Geology and Paleoclimatology Research', 2388],
 ['Particle physics theoretical and experimental studies', 2014],
 ['Astro and Planetary Science', 1638],
 ['Stellar, planetary, and galactic studies', 1355],
 ['Geological and Geochemical Analysis', 1278],
 ['Dental Implant Techniques and Outcomes', 1237],
 ['Climate variability and models', 1230],
 ['Quantum Chromodynamics and Particle Interactions', 1113],
 ['Acute Ischemic Stroke Management', 760],
 ['Schizophrenia research and treatment', 718],
 ['Animal Behavior and Reproduction', 701],
 ['Dental materials and restorations', 692],
 ['Ecology and Vegetation Dynamics Studies', 665],
 ['Cardiac Valve Diseases and Treatments', 635],
 ['Cardiac electrophysiology and arrhythmias', 611],
 ['Oral microbiology and periodontitis research', 596],
 ['Meta-analysis and systematic reviews', 560],
 ['Solar and Space Plasma Dynamics', 556],
 ['Advanced MRI Techniques and Applications', 551],
 ['Cryospheric studies and observations', 516]]

In [55]:
topics_data =[[i['key_display_name'], i['count']] for i in data['group_by']]
topics_sum = 0
for i in topics_data:
    topics_sum += i[1]
topics_sum

67469

In [56]:
# Calculating and adding percentages
for i in topics_data:
    i.append(str(round(i[1]/topics_sum*100,1)) + '%')
topics_data[:10]

[['Geology and Paleoclimatology Research', 2388, '3.5%'],
 ['Particle physics theoretical and experimental studies', 2014, '3.0%'],
 ['Astro and Planetary Science', 1638, '2.4%'],
 ['Stellar, planetary, and galactic studies', 1355, '2.0%'],
 ['Geological and Geochemical Analysis', 1278, '1.9%'],
 ['Dental Implant Techniques and Outcomes', 1237, '1.8%'],
 ['Climate variability and models', 1230, '1.8%'],
 ['Quantum Chromodynamics and Particle Interactions', 1113, '1.6%'],
 ['Acute Ischemic Stroke Management', 760, '1.1%'],
 ['Schizophrenia research and treatment', 718, '1.1%']]

## 👨‍🎓  Authors endpoint

Similar to the *Works* endpoint, there is an ***Authors*** endpoint in OpenAlex, delivering rich information about authors of scholarly publications: `/authors`. The *Authors* endpoint can be accessed via OpenAlex identifier (starting with an "A" like `A2156811675`) or for instance, via the popular [ORCID](https://orcid.org/) identifier. For detailed information see the *Author* object documentation [here](https://developers.openalex.org/api-reference/authors).


In [57]:
base_url = 'https://api.openalex.org/authors/'
params = {'api_key': api_key} 

# Usable identifiers:
openalex_id = 'A2156811675'
openalex_id_namespace = 'openalex:A2156811675'
orcid = 'orcid:0000-0002-9344-1029'

# Adjust the used identifier
r = requests.get(base_url + orcid, params=params) 
data = r.json()

In [58]:
list(data)

['id',
 'orcid',
 'observed_orcids',
 'display_name',
 'raw_author_names',
 'full_name',
 'works_count',
 'cited_by_count',
 'summary_stats',
 'ids',
 'affiliations',
 'last_known_institutions',
 'topics',
 'topic_share',
 'x_concepts',
 'counts_by_year',
 'display_name_alternatives',
 'block_key',
 'works_api_url',
 'updated_date',
 'created_date']

In [59]:
data['id']

'https://openalex.org/A5015900970'

In [60]:
# All available person identifiers
data['ids']

{'openalex': 'https://openalex.org/A5015900970',
 'orcid': 'https://orcid.org/0000-0002-9344-1029',
 'observed_orcids': ['https://orcid.org/0000-0002-9344-1029']}

In [61]:
# Normalized plain name and variants
print(data['display_name'])
data['display_name_alternatives']

Christine L. Borgman


['Borgman Christine',
 'Borgman Christine L',
 'Borgman, C L',
 'Borgman, Christine',
 'Borgman, Christine L',
 'Borgman, Christine L.',
 'C L Borgman',
 'C. Borgman',
 'C. L. Borgman',
 'CHRISTINE L. BORGMAN',
 'Christine Borgman',
 'Christine L Borgman',
 'Christine L. BORGMAN',
 'Christine L. Borgman',
 'Christine L. Borgman (chair',
 'Christine L. Borgman;',
 'Members Christine Borgman']

In [62]:
# How many times the publications of this person were cited?
print(data['cited_by_count'])
data['summary_stats']

13185


{'2yr_mean_citedness': 0.9285714285714286, 'h_index': 52, 'i10_index': 134}

In [63]:
# Count of published works and citations per year, sicne 2012
data['counts_by_year']

[{'year': 1975, 'works_count': 1, 'oa_works_count': 0, 'cited_by_count': 1},
 {'year': 1977, 'works_count': 2, 'oa_works_count': 0, 'cited_by_count': 6},
 {'year': 1978, 'works_count': 1, 'oa_works_count': 0, 'cited_by_count': 1},
 {'year': 1979, 'works_count': 5, 'oa_works_count': 1, 'cited_by_count': 2},
 {'year': 1980, 'works_count': 3, 'oa_works_count': 1, 'cited_by_count': 4},
 {'year': 1981, 'works_count': 3, 'oa_works_count': 0, 'cited_by_count': 2},
 {'year': 1982, 'works_count': 4, 'oa_works_count': 0, 'cited_by_count': 12},
 {'year': 1983, 'works_count': 13, 'oa_works_count': 1, 'cited_by_count': 271},
 {'year': 1984, 'works_count': 7, 'oa_works_count': 0, 'cited_by_count': 87},
 {'year': 1985, 'works_count': 12, 'oa_works_count': 2, 'cited_by_count': 69},
 {'year': 1986, 'works_count': 12, 'oa_works_count': 4, 'cited_by_count': 593},
 {'year': 1987, 'works_count': 7, 'oa_works_count': 1, 'cited_by_count': 81},
 {'year': 1988, 'works_count': 9, 'oa_works_count': 0, 'cited_by_

In [64]:
# API link to the publications of the author (using filter of the "Works" endpoint)
data['works_api_url']

'https://api.openalex.org/works?filter=author.id:A5015900970'

In [65]:
# The person's last known affiliations - a list, since authors can have several (None, if unknown)
print(data['last_known_institutions'])

# All affiliations over time, with the years of affiliated publications
[[i['institution']['display_name'], i['years']] for i in data['affiliations'][:5]]

None


[['Online Computer Library Center', [1983]],
 ['University of California, Los Angeles',
  [2025,
   2023,
   2022,
   2021,
   2020,
   2019,
   2018,
   2017,
   2016,
   2015,
   2014,
   2013,
   2012,
   2011,
   2010,
   2009,
   2008,
   2007,
   2006,
   2005,
   2004,
   2003,
   2002,
   2001,
   2000,
   1999,
   1997,
   1996,
   1995,
   1994,
   1992,
   1991,
   1990,
   1989,
   1988,
   1987,
   1986,
   1985,
   1983]],
 ['The University of Texas at Dallas', [1980]],
 ['University of Washington', [2006]],
 ['University of California, Irvine', [2014]]]

In [66]:
# Topics the author is publishing about (replacing the deprecated "x_concepts")
[[i['display_name'], i['count']] for i in data['topics'][:10]]

[['Research Data Management Practices', 269],
 ['Library Science and Information', 187],
 ['Data Analysis and Archiving', 164],
 ['Scientific Computing and Data Management', 102],
 ['Library Science and Information Literacy', 94]]

### Use the search parameter

Using the search parameter on the `/authors`endpoint looks for the terms in the `display_name` attribute only. Read more about the search functionality [here](https://docs.openalex.org/api-entities/authors/search-authors).

In [67]:
base_url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'search': 'erik eriksen'}

r = requests.get(base_url, params=params)
data = r.json()

In [68]:
data['meta']

{'count': 31,
 'db_response_time_ms': 17,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get authors where text has (erik eriksen)',
  'x_new_oql': 'get authors where text has (erik eriksen)',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'text.search',
     'value': 'erik eriksen',
     'operator': 'has'}]},
  'url': '/authors?filter=text.search:erik eriksen',
  'order': 'best matches first',
  'order_default': True},
 'cost_usd': 0.001}

In [69]:
# Name, number of works and last known affiliations of "erik eriksen" authors
[[i['display_name'], i['works_count'], [j['display_name'] for j in i['last_known_institutions'] or []]] \
                     for i in data['results'][:10]]

[['Erik Fink Eriksen',
  371,
  ['University of Oslo', 'SpesialistSenteret Pilestredet Park']],
 ['Erik Oddvar Eriksen', 154, ['University of Oslo']],
 ['Tor Erik Eriksen', 77, ['Norwegian Institute for Water Research']],
 ['Erik Eriksen', 10, ['University of Oslo']],
 ['Erik Nymann Eriksen', 27, []],
 ['E F Eriksen', 17, []],
 ['Erik A. Eriksen', 18, ['ExxonMobil (United States)']],
 ['Erik F. Eriksen', 1, ['Aarhus University Hospital']],
 ['Erik Eriksen', 10, ['Oslo University Hospital']],
 ['Erik Fink-Eriksen', 1, ['Aarhus University Hospital']]]

### Access multiple Author entities

Like for publications (aka *Works* entities), you can search for mutliple authors at once, and **use filters and groupings**. 

#### Filter authors by institution & put the data into a dataframe

For example, you might search for all authors, whose last known affiliations include *University of Bern*. For this, you can use its OpenAlex identifier `I118564535`. Filtering authors will use the `works_count` attribute to sort the results by default. To adjust the default sorting methods look [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/sort-entity-lists).

In [70]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'last_known_institutions.id:I118564535',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()

In [71]:
data['meta']

{'count': 16808,
 'db_response_time_ms': 30,
 'page': 1,
 'per_page': 20,
 'groups_count': None,
 'x_query': {'oql': 'get authors where last known institution is (I118564535)',
  'x_new_oql': 'get authors where last known institution is (I118564535)',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'last_known_institutions.id',
     'value': 'I118564535'}]},
  'url': '/authors?filter=last_known_institutions.id:I118564535&per_page=20',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

By reading out certain attributes of the result set you can easily transform the data into a dataframe for further research.

In [72]:
names = []
ids = []
orcids = []
workcounts = []
citedcounts = []
workslinks = []

for i in data['results']:
    names.append(i['display_name'])
    ids.append(i['id'])
    orcids.append(i['orcid'])
    workcounts.append(i['works_count'])
    citedcounts.append(i['cited_by_count'])
    workslinks.append(i['works_api_url'])
    
zipped = list(zip(names, ids, orcids, workcounts, citedcounts, workslinks))
df = pd.DataFrame(zipped, columns=['author', 'openalex_id', 'orcid', \
                                    'works_count', 'cited_by_count', 'works_link'])

In [73]:
df

,author,openalex_id,orcid,works_count,cited_by_count,works_link
0,Justin E Albert,https://openalex.org/A5017266796,https://orcid.org/0000-0003-0253-2505,5380,144456,https://api.openalex.org/works?filter=author.i...
1,Mathieu Avanzi,https://openalex.org/A5000267880,https://orcid.org/0000-0002-0688-2310,4327,2440,https://api.openalex.org/works?filter=author.i...
2,Antonio Ereditato,https://openalex.org/A5079313387,https://orcid.org/0000-0002-5423-8079,4256,138392,https://api.openalex.org/works?filter=author.i...
3,T. Dai,https://openalex.org/A5110190636,https://orcid.org/0000-0002-9607-5124,3752,123373,https://api.openalex.org/works?filter=author.i...
4,Y. Haddad,https://openalex.org/A5106613391,https://orcid.org/0000-0003-4916-7752,3490,39765,https://api.openalex.org/works?filter=author.i...
5,Elin Lundstad,https://openalex.org/A5063753198,https://orcid.org/0000-0002-3443-2480,2694,405,https://api.openalex.org/works?filter=author.i...
6,S. Francescato,https://openalex.org/A5044213178,https://orcid.org/0000-0001-5315-9275,2149,34640,https://api.openalex.org/works?filter=author.i...
7,Michael John Zaworotko,https://openalex.org/A5001382768,https://orcid.org/0000-0002-1360-540X,1581,63244,https://api.openalex.org/works?filter=author.i...
8,J. A. Aguilar–Saavedra,https://openalex.org/A5046527339,https://orcid.org/0000-0002-5475-8920,1513,115939,https://api.openalex.org/works?filter=author.i...
9,G. Glemža,https://openalex.org/A5087923387,https://orcid.org/0000-0003-4977-5256,1285,3575,https://api.openalex.org/works?filter=author.i...


#### Filter by multiple features & use logical operators

You can not only use several filters at once, by **simply stacking** them. An you can use filters with **logical operators**, like `!` for NOT, `|` for OR, less-than `<`, or greater-than `>`. For further details see [this page](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).

In [74]:
# Example: Authors with only a few publications, but high citation count
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'works_count:<5,cited_by_count:>50000',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 12,
 'db_response_time_ms': 21,
 'page': 1,
 'per_page': 20,
 'groups_count': None,
 'x_query': {'oql': 'get authors with a citation count above 50,000 with a works count below 5',
  'x_new_oql': 'get authors with a citation count above 50,000 with a works count below 5',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'cited_by_count',
     'value': 50000,
     'operator': '>'},
    {'column_id': 'works_count', 'value': 5, 'operator': '<'}]},
  'url': '/authors?filter=cited_by_count:%3E50000,works_count:%3C5&per_page=20',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [75]:
[[i['display_name'], i['works_count'], i['cited_by_count'], i['works_api_url']] for i in data['results']]

[['T. Meyarivan',
  4,
  55762,
  'https://api.openalex.org/works?filter=author.id:A5066045362'],
 ['Helge Krieg',
  3,
  59606,
  'https://api.openalex.org/works?filter=author.id:A5011411945'],
 ['Andrew Markiel',
  2,
  51784,
  'https://api.openalex.org/works?filter=author.id:A5031053426'],
 ['NiraJ. Rosebrough',
  2,
  317744,
  'https://api.openalex.org/works?filter=author.id:A5032482932'],
 ['S. Nicklen',
  2,
  68758,
  'https://api.openalex.org/works?filter=author.id:A5035256244'],
 ['Xuelei Xie',
  2,
  54136,
  'https://api.openalex.org/works?filter=author.id:A5037416293'],
 ['Caroline A Schneider',
  2,
  64518,
  'https://api.openalex.org/works?filter=author.id:A5111881458'],
 ['Kalpit Shah',
  2,
  57062,
  'https://api.openalex.org/works?filter=author.id:A5141546693'],
 ['Smruti Patel',
  2,
  57062,
  'https://api.openalex.org/works?filter=author.id:A5141763029'],
 ['Nathaniel Gonzalez',
  1,
  51273,
  'https://api.openalex.org/works?filter=author.id:A5057407051'],
 ['R

In [76]:
# Example: Authors from companies or nonprofit institutions outside the USA and China
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'last_known_institutions.type:company|nonprofit,last_known_institutions.country_code:!US|CN',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 1288495,
 'db_response_time_ms': 96,
 'page': 1,
 'per_page': 20,
 'groups_count': None,
 'x_query': {'oql': 'get authors where author country is not ([China](CN) or [United States](US)) and institution type is ([company](company) or [nonprofit](nonprofit))',
  'x_new_oql': 'get authors where author country is not ([China](CN) or [United States](US)) and institution type is ([company](company) or [nonprofit](nonprofit))',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'last_known_institutions.country_code',
     'value': 'CN',
     'is_negated': True},
    {'column_id': 'last_known_institutions.country_code',
     'value': 'US',
     'is_negated': True},
    {'join': 'or',
     'filters': [{'column_id': 'last_known_institutions.type',
       'value': 'company'},
      {'column_id': 'last_known_institutions.type', 'value': 'nonprofit'}]}]},
  'url': '/authors?filter=last_known_institutions.country_code:%21CN,last_known_institutions.country_code:%21US,last_kn

In [77]:
[[i['display_name'], [j['display_name'] for j in i['last_known_institutions'] or []]] for i in data['results']]

[['T. Kobayashi',
  ['Kyushu University',
   'Hamamatsu Photonics (Japan)',
   'National Institutes of Natural Sciences',
   'The Graduate University for Advanced Studies, SOKENDAI',
   'National Institute for Fusion Science',
   'Research Institute for Applied Mechanics, Kyushu University']],
 ['N. Tamura',
  ['Max Planck Society',
   'National Institute for Fusion Science',
   'Max Planck Institute for Plasma Physics - Greifswald',
   'Max Planck Institute for Plasma Physics']],
 ['A. Shimizu',
  ['National Institutes of Natural Sciences',
   'The Graduate University for Advanced Studies, SOKENDAI',
   'National Institute for Fusion Science',
   'Ushio (Japan)']],
 ['Yoshinori Mizuno',
  ['National Institutes of Natural Sciences',
   'The Graduate University for Advanced Studies, SOKENDAI',
   'National Institute for Fusion Science',
   'Asahi Kogyosha Co., Ltd. (Japan)']],
 ['Alexander K. C. Leung',
  ['University of Calgary',
   "Alberta Children's Hospital",
   'Alberta Children’s

#### Group authors due to a certain feature

As a use case, it is very easy to group all authors according to their last known institution's country. Just use the `last_known_institutions.country_code` sub-attribute of the *Author* object as **grouping parameter**. Find more features to group authors by [here](https://docs.openalex.org/api-entities/authors/group-authors).

In [78]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'group_by': 'last_known_institutions.country_code'}

r = requests.get(url, params=params) 
data = r.json()

In [79]:
data['meta']

{'count': 96816014,
 'db_response_time_ms': 2,
 'page': 1,
 'per_page': 200,
 'groups_count': 200,
 'x_query': {'oql': 'get authors; then, group those authors by institution country',
  'x_new_oql': 'get authors; then, group those authors by institution country',
  'oqo': {'get_rows': 'authors',
   'group_by': [{'column_id': 'last_known_institutions.country_code'}]},
  'url': '/authors?group_by=last_known_institutions.country_code',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [80]:
# Accessing the 10 biggest countries of affiliations, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:10]]

[['China', 6580250],
 ['United States', 6289678],
 ['Indonesia', 3148506],
 ['India', 2018613],
 ['Japan', 1887456],
 ['Brazil', 1769787],
 ['United Kingdom', 1323158],
 ['Germany', 1322575],
 ['Russia', 1068271],
 ['France', 923964]]

#### Combine filters and grouping

Of course you can combine filters and groupings also for authors' data. In the example beneath you will group all authors writing about *Ethical Implications of Artificial Intelligence* `T10883` according to their (last known) institution type.

In [81]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'topics.id:T10883',
         'group_by': 'last_known_institutions.type'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 154036,
 'db_response_time_ms': 1,
 'page': 1,
 'per_page': 200,
 'groups_count': 9,
 'x_query': {'oql': 'get authors where topic is (T10883);\nthen, group those authors by institution type',
  'x_new_oql': 'get authors where topic is (T10883);\nthen, group those authors by institution type',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'topics.id', 'value': 'T10883'}],
   'group_by': [{'column_id': 'last_known_institutions.type'}]},
  'url': '/authors?filter=topics.id:T10883&group_by=last_known_institutions.type',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [82]:
data['group_by']

[{'key': 'education', 'key_display_name': 'education', 'count': 74490},
 {'key': 'facility', 'key_display_name': 'facility', 'count': 3757},
 {'key': 'company', 'key_display_name': 'company', 'count': 3611},
 {'key': 'healthcare', 'key_display_name': 'healthcare', 'count': 2525},
 {'key': 'government', 'key_display_name': 'government', 'count': 2307},
 {'key': 'nonprofit', 'key_display_name': 'nonprofit', 'count': 1839},
 {'key': 'other', 'key_display_name': 'other', 'count': 844},
 {'key': 'funder', 'key_display_name': 'funder', 'count': 181},
 {'key': 'archive', 'key_display_name': 'archive', 'count': 93}]

In [83]:
inst_data =[[i['key_display_name'], i['count']] for i in data['group_by']]
inst_sum = 0
for i in inst_data:
    inst_sum += i[1]
inst_sum

89647

In [84]:
# Calculating and adding percentages
for i in inst_data:
    i.append(str(round(i[1]/inst_sum*100,1)) + '%')
inst_data

[['education', 74490, '83.1%'],
 ['facility', 3757, '4.2%'],
 ['company', 3611, '4.0%'],
 ['healthcare', 2525, '2.8%'],
 ['government', 2307, '2.6%'],
 ['nonprofit', 1839, '2.1%'],
 ['other', 844, '0.9%'],
 ['funder', 181, '0.2%'],
 ['archive', 93, '0.1%']]

Now let's have a look at those authors writing about AI Ethics, whose last know affiliation is a company. For this, we will use 2 filters concurrently here: `topics.id:T10883` and `last_known_institutions.type:company`.

In [85]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'topics.id:T10883,last_known_institutions.type:company'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 3611,
 'db_response_time_ms': 52,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get authors where institution type is [company](company) and topics is (T10883)',
  'x_new_oql': 'get authors where institution type is [company](company) and topics is (T10883)',
  'oqo': {'get_rows': 'authors',
   'filter_rows': [{'column_id': 'last_known_institutions.type',
     'value': 'company'},
    {'column_id': 'topics.id', 'value': 'T10883'}]},
  'url': '/authors?filter=last_known_institutions.type:company,topics.id:T10883',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [86]:
# Showing the 20 most productive authors in AI Ethics working at a company
[[i['display_name'], [j['display_name'] for j in i['last_known_institutions'] or []], i['works_count']] \
                     for i in data['results'][:20]]

[['Gemini 3.1 (Flash)', ['Google DeepMind (United Kingdom)'], 2220],
 ['Oleh Ivchenko',
  ['Odesа Polytechnic National University',
   'Capgemini (United States)',
   'State University of Intelligent Technologies and Telecommunications'],
  1096],
 ['Luciano Floridi',
  ['Yale University',
   'Data Power Decisions (United States)',
   'University of Bologna'],
  879],
 ['Hamed Taherdoost',
  ['University Canada West',
   'Global University Systems',
   'National Kaohsiung University of Science and Technology',
   'Westcliff University',
   'RMIT University'],
  450],
 ['Michael J. Muller', ['IBM Research - Cambridge (United States)'], 387],
 ['Narnaiezzsshaa Truong', ['Sofar Ocean (United States)'], 374],
 ['Natali Helberger',
  ['Information Visualization and Innovative Research',
   'University of Amsterdam'],
  372],
 ['Daniele Quercia',
  ['Politecnico di Torino', "King's College London", 'Nokia (United Kingdom)'],
  356],
 ['Nanyun Peng',
  ['Google (United States)',
   'Pennsylva

## 🔖 Topics endpoint

***Topics*** entities are OpenAlex' notion of subjects, which are attached to *Works*, using an automated system that takes into account the available information including title, abstract, source (journal) name, and citations. The highest-scoring topic is that Work's `primary_topic`. There is also an attribute `topics`of a Work entity with additional highly ranked topics.

Topics are hierarchically organzied in sub-fields, fields, and domains, for further information see the [Topics page](https://docs.openalex.org/api-entities/topics)


In [87]:
base_url = 'https://api.openalex.org/topics/'
params = {'api_key': api_key} 

openalex_id = 'T10101'

r = requests.get(base_url + openalex_id, params=params) 
data = r.json()

In [88]:
# Available attributes of the Topic entity
list(data)

['id',
 'display_name',
 'description',
 'keywords',
 'legacy_keywords',
 'ids',
 'subfield',
 'field',
 'domain',
 'siblings',
 'works_count',
 'cited_by_count',
 'works_api_url',
 'updated_date',
 'created_date']

In [89]:
print(data['display_name'])
print(data['ids'])
data['keywords']

Cloud Computing and Resource Management
{'openalex': 'https://openalex.org/T10101', 'wikipedia': 'https://en.wikipedia.org/wiki/Cloud_computing'}


[{'id': 'https://openalex.org/keywords/cloud-computing',
  'display_name': 'cloud computing',
  'score': 0.3186},
 {'id': 'https://openalex.org/keywords/virtualization',
  'display_name': 'virtualization',
  'score': 0.382},
 {'id': 'https://openalex.org/keywords/virtual-machines',
  'display_name': 'virtual machines',
  'score': 0.3669},
 {'id': 'https://openalex.org/keywords/resource-utilization',
  'display_name': 'resource utilization',
  'score': 0.2206},
 {'id': 'https://openalex.org/keywords/mapreduce',
  'display_name': 'MapReduce',
  'score': 0.3779},
 {'id': 'https://openalex.org/keywords/hadoop',
  'display_name': 'Hadoop',
  'score': 0.416},
 {'id': 'https://openalex.org/keywords/virtual-machine-placement',
  'display_name': 'virtual machine placement',
  'score': 0.8807},
 {'id': 'https://openalex.org/keywords/service-level-agreements',
  'display_name': 'service level agreements',
  'score': 0.2981},
 {'id': 'https://openalex.org/keywords/microsoft-azure',
  'display_name

In [90]:
data['description']

'This cluster of papers covers a wide range of topics related to cloud computing, big data technologies, and data center management. It includes discussions on MapReduce, virtualization, Hadoop, resource management, energy efficiency, scalability, and distributed systems. The papers also address challenges and opportunities in the field of cloud computing and big data.'

## 🏫 Institutions endpoint

***Institution*** entities are the organizations which host the authors when they write their publications (affiliations). Thus, publications can be assigned as output to certain institutions. OpenAlex uses the categories of institutions of the [ROR](https://ror.org/) registry: education, funder, healthcare, company, archive, nonprofit, government, facility, other. See also the [Institution](https://docs.openalex.org/about-the-data/institution) object documentation.

You can use the `/institutions` endpoint with OpenAlex identifier (starting with an "I" like `I118564535`) or the popular identifier ROR ID.

In [91]:
base_url = 'https://api.openalex.org/institutions/'
params = {'api_key': api_key} 

#openalex_id = 'I118564535'
#openalex_id_namespace = 'openalex:I118564535'
ror_id = 'ror:02k7v4d05'

# Adapt identifier if necessary
r = requests.get(base_url + ror_id, params=params) 
data = r.json()

In [92]:
# Overview of data fields
list(data)

['id',
 'ror',
 'display_name',
 'country_code',
 'type',
 'type_id',
 'lineage',
 'homepage_url',
 'image_url',
 'image_thumbnail_url',
 'display_name_acronyms',
 'display_name_alternatives',
 'repositories',
 'works_count',
 'cited_by_count',
 'summary_stats',
 'ids',
 'geo',
 'international',
 'associated_institutions',
 'counts_by_year',
 'roles',
 'topics',
 'topic_share',
 'is_super_system',
 'status',
 'works_api_url',
 'updated_date',
 'created_date']

In [93]:
# OpenAlex institution ID
data['id']

'https://openalex.org/I118564535'

In [94]:
# All institution identifiers
data['ids']

{'openalex': 'https://openalex.org/I118564535',
 'ror': 'https://ror.org/02k7v4d05',
 'grid': 'grid.5734.5',
 'wikipedia': 'http://en.wikipedia.org/wiki/University_of_Bern',
 'wikidata': None}

In [95]:
# Preferable name
data['display_name']

'University of Bern'

In [96]:
# Display name variants, e.g. in other languages
data['display_name_alternatives']

['University of Bern',
 'Università di Berna',
 'Universität Bern',
 'Université de Berne']

In [97]:
# API link to all affiliated publications
data['works_api_url']

'https://api.openalex.org/works?filter=institutions.id:I118564535'

In [98]:
# Number of citations of affiliated publications
data['cited_by_count']

15852181

In [99]:
# Number of affiliated publications and citations by year, since 2012
data['counts_by_year']

[{'year': 10000,
  'works_count': 1825,
  'oa_works_count': 1825,
  'cited_by_count': 5285},
 {'year': 2035, 'works_count': 3, 'oa_works_count': 3, 'cited_by_count': 0},
 {'year': 2027, 'works_count': 3, 'oa_works_count': 3, 'cited_by_count': 0},
 {'year': 2026,
  'works_count': 8697,
  'oa_works_count': 7788,
  'cited_by_count': 4948},
 {'year': 2025,
  'works_count': 139585,
  'oa_works_count': 138305,
  'cited_by_count': 40494},
 {'year': 2024,
  'works_count': 10290,
  'oa_works_count': 8982,
  'cited_by_count': 76151},
 {'year': 2023,
  'works_count': 9740,
  'oa_works_count': 8521,
  'cited_by_count': 121747},
 {'year': 2022,
  'works_count': 9979,
  'oa_works_count': 8341,
  'cited_by_count': 166049},
 {'year': 2021,
  'works_count': 9324,
  'oa_works_count': 7832,
  'cited_by_count': 228820},
 {'year': 2020,
  'works_count': 8005,
  'oa_works_count': 6416,
  'cited_by_count': 321782},
 {'year': 2019,
  'works_count': 6995,
  'oa_works_count': 5039,
  'cited_by_count': 270547},


In [100]:
# Associated institutions
data['associated_institutions']

[{'id': 'https://openalex.org/I2801112126',
  'ror': 'https://ror.org/01q9sj412',
  'display_name': 'University Hospital of Bern',
  'country_code': 'CH',
  'type': 'healthcare',
  'relationship': 'related'},
 {'id': 'https://openalex.org/I4210087665',
  'ror': 'https://ror.org/003bz8x96',
  'display_name': 'Wiederkäuerklinik',
  'country_code': 'CH',
  'type': 'healthcare',
  'relationship': 'related'},
 {'id': 'https://openalex.org/I4387152480',
  'ror': 'https://ror.org/03dz0k314',
  'display_name': 'Wyss Academy for Nature',
  'country_code': None,
  'type': 'education',
  'relationship': 'child'}]

In [101]:
# Geographical information
data['geo']

{'city': 'Bern',
 'geonames_city_id': '2661552',
 'region': 'Bern',
 'country_code': 'CH',
 'country': 'Switzerland',
 'latitude': 46.94809,
 'longitude': 7.44744}

### Access multiple Institution entities

You can access all *Institutions* at once, and use filters and groupings on them. See the available filters for *Institution* entities [here](https://docs.openalex.org/api-entities/institutions/filter-institutions) and possible grouping parameters [here](https://docs.openalex.org/api-entities/institutions/group-institutions).

In the example you will narrow down to Swiss institutions only and group these due to institution types.

In [102]:
url = 'https://api.openalex.org/institutions?'
params = {'api_key': api_key,
         'filter': 'country_code:CH',
         'group_by': 'type'}

r = requests.get(url, params=params) 
data = r.json()

In [103]:
data['meta']

{'count': 1970,
 'db_response_time_ms': 1,
 'page': 1,
 'per_page': 200,
 'groups_count': 9,
 'x_query': {'oql': 'get institutions where country is [Switzerland](CH);\nthen, group those institutions by institution type',
  'x_new_oql': 'get institutions where country is [Switzerland](CH);\nthen, group those institutions by institution type',
  'oqo': {'get_rows': 'institutions',
   'filter_rows': [{'column_id': 'country_code', 'value': 'CH'}],
   'group_by': [{'column_id': 'type'}]},
  'url': '/institutions?filter=country_code:CH&group_by=type',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [104]:
[[i['key_display_name'], i['count']] for i in data['group_by']]

[['company', 647],
 ['nonprofit', 347],
 ['facility', 204],
 ['other', 193],
 ['healthcare', 173],
 ['education', 160],
 ['government', 114],
 ['archive', 105],
 ['funder', 27]]

Now, let's look at the Swiss archives producing scholarly publications. To get this data, you have to use 2 filters at once: `country_code:CH` and `type:archive`.

In [105]:
url = 'https://api.openalex.org/institutions?'
params = {'api_key': api_key,
         'filter': 'country_code:CH,type:archive'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 105,
 'db_response_time_ms': 38,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get institutions in [Switzerland](CH) where type is (archive)',
  'x_new_oql': 'get institutions in [Switzerland](CH) where type is (archive)',
  'oqo': {'get_rows': 'institutions',
   'filter_rows': [{'column_id': 'country_code', 'value': 'CH'},
    {'column_id': 'type', 'value': 'archive'}]},
  'url': '/institutions?filter=country_code:CH,type:archive',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [106]:
# Showing the top 10
[[i['display_name'], i['works_count']] for i in data['results'][:20]]

[['Natural History Museum of Geneva', 4596],
 ['Natural History Museum of Basel', 2817],
 ['Jardin Botanique de Genève', 2760],
 ['Natural History Museum of Bern', 1703],
 ['info fauna', 1044],
 ['Naturéum', 965],
 ['Natur Museum Luzern', 639],
 ['Sukkulenten-Sammlung Zürich', 292],
 ['Swiss National Park', 261],
 ['Museo Cantonale di Storia Naturale di Lugano', 160],
 ['Zürich Zoological Garden', 153],
 ['Swiss National Museum', 139],
 ['Lib4RI - Library for the Research Institutes within the ETH Domain: Eawag, Empa, PSI & WSL',
  95],
 ['Laténium', 93],
 ['Naturmuseum Solothurn', 91],
 ['Zentral und Hochschulbibliothek Luzern', 87],
 ['Cantonal and University Library of Lausanne', 70],
 ['Musée cantonal de géologie', 66],
 ["Musée Cantonal d'Archéologie et d'Histoire", 57],
 ['Swiss National Library', 53]]

## 📰 Sources endpoint

***Sources*** entities in OpenAlex are the **journals, conferences, preprint repositories**, and **institutional repositories** that hold the single publications aka `Works`. For journal articles this will be their journals, for book parts their volume. Sources for instance can also be monograph series like `S4210167446` or repositories like `S2734324842`.

To access data about a certain *Source*, you can use its OpenAlex identifier (starting with a "S" like [S4210209919](https://openalex.org/S4210209919)) or the the popular [ISSN](https://en.wikipedia.org/wiki/International_Standard_Serial_Number) identifier. See also the [Source](https://docs.openalex.org/api-entities/sources) entity documentation and try a random source with `https://api.openalex.org/sources/random`.

In [107]:
base_url = 'https://api.openalex.org/sources/'
params = {'api_key': api_key} 

#openalex_id = 'S98348390'
#issn = 'issn:0028-0836'   # non-OA journal
issn = 'issn:1654-6369'   # OA journal

r = requests.get(base_url + issn, params=params) 
data = r.json()

In [108]:
list(data)

['id',
 'issn_l',
 'issn',
 'display_name',
 'host_organization',
 'host_organization_name',
 'host_organization_lineage',
 'works_count',
 'oa_works_count',
 'cited_by_count',
 'summary_stats',
 'is_oa',
 'is_in_doaj',
 'is_in_doaj_since_year',
 'is_high_oa_rate',
 'is_high_oa_rate_since_year',
 'is_in_scielo',
 'is_ojs',
 'is_preprint_repository',
 'is_core',
 'listed_in',
 'oa_flip_year',
 'first_publication_year',
 'last_publication_year',
 'ids',
 'homepage_url',
 'apc_prices',
 'apc_usd',
 'apc_usd_by_year',
 'country_code',
 'societies',
 'alternate_titles',
 'type',
 'topics',
 'topic_share',
 'counts_by_year',
 'works_api_url',
 'updated_date',
 'created_date']

In [109]:
# All identifiers. "issn_l" should be used preferably while using OpenAlex API.
data['ids']

{'openalex': 'https://openalex.org/S107893744',
 'issn_l': '1654-4951',
 'issn': ['1654-4951', '1654-6369'],
 'mag': '107893744',
 'wikidata': None}

In [110]:
# Title of the source & name of host organization/publisher & website
print(data['display_name'])
print(data['host_organization_name'])
data['homepage_url']

Ethics & Global Politics
Taylor & Francis


'https://www.tandfonline.com/toc/zegp20/current'

In [111]:
# Is it an Open Access venue (and is it in DOAJ registry)?
print(data['is_oa'])
data['is_in_doaj']

True


True

In [112]:
# Number of hosted publications
data['works_count']

292

In [113]:
# Topics
[[i['display_name'] for i in data['topics']]]

[['Political Philosophy and Ethics',
  'Philosophical Ethics and Theory',
  'Migration, Refugees, and Integration',
  'Political Theory and Democracy',
  'International Relations and Foreign Policy',
  'Migration and Exile Studies',
  'War, Ethics, and Justification',
  'Legal Rights and Human Rights',
  'International Law and Human Rights',
  'Critical Theory and Philosophy',
  'Globalization and political ideologies',
  'Critical Theory and Political Philosophy',
  'Feminism, Gender, and Intersectionality',
  'Development, Ethics, and Society',
  'Migration, Racism, and Human Rights',
  'Migration and Labor Dynamics',
  'Immigration Law and Human Rights',
  'Education, Philosophy, and Society',
  'Political Theology and Sovereignty',
  'Feminist Epistemology and Gender Studies',
  'Global Education and Multiculturalism',
  'Global Peace and Security Dynamics',
  'Kantian Philosophy and Modern Interpretations',
  'Decolonial Thought and Epistemologies',
  'Evolving Legal Systems and G

In [114]:
# APC information
data['apc_prices']

[]

In [115]:
# API link to hosted publications
data['works_api_url']

'https://api.openalex.org/works?filter=primary_location.source.id:S107893744'

### Access multiple Source entities

You can access all *Sources* at once, and use filters and groupings on them. See the available filters for *Source* entities [here](https://docs.openalex.org/api-entities/sources/filter-sources) and possible grouping parameters [here](https://docs.openalex.org/api-entities/sources/group-sources).

In the example at the end you will narrow down to open access sources only and group these due to publishers resp. `host_organization`.

In [116]:
# Acessing all sources (sorted by number of works)
url = 'https://api.openalex.org/sources?'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 269018,
 'db_response_time_ms': 19,
 'page': 1,
 'per_page': 25,
 'groups_count': None,
 'x_query': {'oql': 'get sources',
  'x_new_oql': 'get sources',
  'oqo': {'get_rows': 'sources'},
  'url': '/sources',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [117]:
# Showing the first 20 biggest sources (name, number of works, OA status)
[[i['display_name'], i['works_count'], i['is_oa']] for i in data['results'][:20]]

[['NIFS', 43537371, False],
 ['DiSSCo', 11171533, False],
 ['PubMed', 7943796, False],
 ['Zenodo (CERN European Organization for Nuclear Research)', 6435567, True],
 ['Global Biodiversity Information Facility', 4725445, False],
 ['Legacy IGSN ID re-registration for Geoscience Australia SAMPLES data',
  4418575,
  False],
 ['SESAR_ODP', 3915011, False],
 ['Institutional Repositories DataBase (IRDB)', 3625875, False],
 ['Plutof.  Data Management and Publishing Platform', 3571818, False],
 ['Springer eBooks', 3439209, False],
 ['IGSN ID registration for Geoscience SAMPLES data', 3127452, False],
 ['University of Michigan Library Repository', 2993385, False],
 ['Figshare', 2857725, False],
 ['HAL', 2195706, False],
 ['LA Referencia (Red Federada de Repositorios Institucionales de Publicaciones Científicas)',
  2147911,
  False],
 ['Dialnet (Universidad de la Rioja)', 2104689, False],
 ['International Treaty on Plant Genetic Resources for Food and Agriculture',
  1976000,
  False],
 ['Routl

In [118]:
# Inspecting OA sources according to their publishers
url = 'https://api.openalex.org/sources?'
params = {'api_key': api_key,
         'filter': 'is_oa:True',
         'group_by': 'host_organization'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

{'count': 76632,
 'db_response_time_ms': 4,
 'page': 1,
 'per_page': 200,
 'groups_count': 200,
 'x_query': {'oql': "get sources where it's fully open access; then, group those sources by publisher",
  'x_new_oql': "get sources where it's fully open access; then, group those sources by publisher",
  'oqo': {'get_rows': 'sources',
   'filter_rows': [{'column_id': 'is_oa', 'value': True}],
   'group_by': [{'column_id': 'host_organization'}]},
  'url': '/sources?filter=is_oa:true&group_by=host_organization',
  'order': 'most works first',
  'order_default': True},
 'cost_usd': 0.0001}

In [119]:
pub_data = [[i['key_display_name'], i['count']] for i in data['group_by']]
pub_sum = 0
for i in pub_data:
    pub_sum += i[1]
pub_sum

15359

In [120]:
# Showing the 20 biggest OA publishers (according to number of sources), with percentages
for i in pub_data:
    i.append(str(round(i[1]/pub_sum*100,1)) + '%')
pub_data[:20]

[['Elsevier BV', 1120, '7.3%'],
 ['Multidisciplinary Digital Publishing Institute', 372, '2.4%'],
 ['Hindawi Publishing Corporation', 312, '2.0%'],
 ['De Gruyter Open', 307, '2.0%'],
 ['BioMed Central', 297, '1.9%'],
 ['Medknow', 287, '1.9%'],
 ['Wiley', 286, '1.9%'],
 ['Science Publishing Group', 270, '1.8%'],
 ['SAGE Publishing', 247, '1.6%'],
 ['Springer Science+Business Media', 244, '1.6%'],
 ['Scientific Research Publishing', 235, '1.5%'],
 ['Taylor & Francis', 234, '1.5%'],
 ['De Gruyter', 226, '1.5%'],
 ['Frontiers Media', 214, '1.4%'],
 ['Springer Nature', 190, '1.2%'],
 ['OMICS Publishing Group', 184, '1.2%'],
 ['African Journals OnLine', 178, '1.2%'],
 ['Egypts Presidential Specialized Council for Education and Scientific Research',
  166,
  '1.1%'],
 ['Bentham Science Publishers', 162, '1.1%'],
 ['Padang State University', 135, '0.9%']]